# Research RAG — Basic Usage

This notebook demonstrates the full Research RAG pipeline:
1. Load configuration
2. Initialize embedding and storage services
3. Ingest PDF documents
4. Run semantic queries
5. Generate citation-grounded answers

All cells use local fallback models when no API key is available, so the notebook is runnable out-of-the-box.

In [ ]:
%%capture
from research_rag.config import load_settings

settings = load_settings()
print(f"Log level: {settings.log_level}")
print(f"Chunk size range: {settings.ingestion.chunk_size_min}-{settings.ingestion.chunk_size_max}")
print(f"Top-k retrieval: {settings.retrieval.top_k}")

In [ ]:
%%capture
from research_rag.embeddings import EmbeddingService

# api_key=None triggers local fallback (BGE-small-en-v1.5)
embed = EmbeddingService(api_key=None)
print(embed)

In [ ]:
%%capture
from pathlib import Path
from research_rag.storage.chroma import ChromaStore

store = ChromaStore(
    persist_directory=Path("./data/chroma"),
    embedding_service=embed,
)
print(f"Chroma store ready. Collection: {store.collection_name}")

## Ingesting PDFs

The `IngestionPipeline` orchestrates parsing, metadata extraction, and chunking.
Point it at a directory of PDFs and it will produce structured chunks ready for the vector store.

Replace `./pdfs/` with the path to your own PDF collection.

In [ ]:
%%capture
from pathlib import Path
from research_rag.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline(
    config=settings.ingestion,
    output_dir=Path("./data"),
)

pdf_dir = Path("./pdfs/")
if pdf_dir.exists():
    results = pipeline.process_directory(pdf_dir)
    success = sum(1 for r in results if r.success)
    total_chunks = sum(r.chunks_created for r in results)
    print(f"Processed {success}/{len(results)} PDFs, {total_chunks} chunks created.")
else:
    print(f"PDF directory not found: {pdf_dir}. Create it and add some PDFs to try ingestion.")

## Querying

Use the `Retriever` to run semantic search over the stored chunks.
Results include relevance scores, page numbers, and excerpt text.

In [ ]:
%%capture
from research_rag.retrieval import Retriever

retriever = Retriever(store=store)
results = retriever.search("partition violence")

if results:
    print(retriever.format_results(results))
else:
    print("No results found. Ingest some documents first.")

## Citation-grounded answers

`AnswerGenerator` retrieves evidence, builds a synthesis prompt, calls an LLM,
and parses citations so every claim is traceable to source documents.

If an `OPENROUTER_API_KEY` is set, it uses the remote model; otherwise it falls back
to a local mode (the exact fallback depends on your environment).

In [ ]:
%%capture
from research_rag.synthesis import AnswerGenerator

generator = AnswerGenerator(
    retriever=retriever,
    top_k=settings.retrieval.top_k,
)

response = generator.answer("What were the main causes of partition violence?")
print(f"Confidence: {response['confidence']:.2f}")
print("\nAnswer:")
print(response["answer"])

if response["citations"]:
    print("\nSources:")
    for c in response["citations"]:
        print(f"  - {c['title']} (p. {c['page']}, relevance {c['relevance_score']:.3f})")